# Example: degenerate data

Real samples contain values that break the assumptions of a smooth mixture: a sensor stuck at one reading, single
outliers, rounding. A component built on such a group is almost a spike; its likelihood is huge, it wins every
comparison, and EM never leaves it. This notebook shows the two checks that protect a start, and what happens
without them.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from pysatl_mpest.initializers import (
    ClusterInitializer,
    InitializationError,
    KMeansPartitioner,
    MultiStartInitializer,
    RandomCentersPartitioner,
)

## A stuck sensor

Two wide normal modes and 40 readings of a sensor stuck near 15000. Some partitions put the stuck readings into a
cluster of their own. The component on it is narrower than `1e-6` of the spread of the sample, so the width check
rejects it; the multistart skips such a run and says so.

In [ ]:
rng = np.random.default_rng(42)
X = np.concatenate([rng.normal(0, 1000, 1000), rng.normal(6000, 1500, 1000), 15000 + rng.uniform(-0.001, 0.001, 40)])
families = [normal, normal]
initializer = ClusterInitializer(RandomCentersPartitioner())

for child in np.random.default_rng(13).spawn(10):
    try:
        initializer.initialize(X, families, random_state=child)
    except InitializationError as error:
        print("skipped run:", error)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    start = MultiStartInitializer(initializer, n_init=10).initialize(X, families, random_state=13)
print("warning:", caught[0].message)
show(start)

What the skipped run would have given: the stuck readings as one component and everything else as the other. Its
likelihood is higher than that of the kept start, and EM from it never finds the two real modes.

In [ ]:
ordinary, stuck = X[:2000], X[2000:]
spike = MixtureModel(
    [normal(mu=ordinary.mean(), sigma=ordinary.std()), normal(mu=stuck.mean(), sigma=stuck.std())],
    weights=[2000 / 2040, 40 / 2040],
)
print("spike start:", round(spike.loglikelihood(X), 1), " kept start:", round(start.loglikelihood(X), 1))
from_spike, from_start = em(30).fit(X, spike), em(30).fit(X, start)
show(from_spike)
show(from_start)

fig, axes = plt.subplots(1, 2, figsize=(16, 3.5), sharey=True)
plot_mixtures(axes[0], X, {"EM from the spike": from_spike}, bins=100)
plot_mixtures(axes[1], X, {"EM from the kept start": from_start}, bins=100)
plt.show()

assert spike.loglikelihood(X) > start.loglikelihood(X)

## An outlier

A small sample with one distant point. With `min_cluster_size=1` k-means gives the outlier a cluster of its own, and
the start has a component on one point; with the default threshold (4 points for `Normal`, twice its two free parameters) the
partition is rejected and a multistart looks for another one.

In [ ]:
rng = np.random.default_rng(42)
small = np.concatenate([rng.normal(0, 1, 20), [100.0]])

lax = ClusterInitializer(KMeansPartitioner(), min_cluster_size=1).initialize(small, [normal, expon], random_state=0)
print("min_cluster_size=1:")
show(lax)
try:
    ClusterInitializer(KMeansPartitioner()).initialize(small, [normal, normal], random_state=0)
except InitializationError as error:
    print("default threshold:", error)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    kept = MultiStartInitializer(ClusterInitializer(RandomCentersPartitioner()), n_init=10).initialize(small, [normal, normal], random_state=0)
print("warning:", caught[0].message)
show(kept)

## Takeaway

Both checks are on by default and need no tuning: the width check catches atoms of almost equal values, the size
threshold catches clusters of a few points. Neither silently replaces a bad component: the run fails, and a
multistart finds another one.